# Upload your catalog to **UTD**Quake 📤
[![Docs](https://img.shields.io/badge/docs-UTDQuake-orange)](https://utdquake.readthedocs.io) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97-Dataset-yellow)](https://huggingface.co/datasets/ecastillot/UTDQuake) [![PyPI](https://img.shields.io/pypi/v/utdquake?color=blue)](https://pypi.org/project/utdquake/) [![GitHub](https://img.shields.io/badge/GitHub-UTDQuake-black?logo=github)](https://github.com/ecastillot/UTDQuake)

Build a UTDQuake network from **100 earthquakes of the
[COSEISMIQ](http://coseismiq.ethz.ch) project** (Hengill geothermal area, Iceland): events and picks from
the COSEISMIQ event service, station locations from ETH Zurich. Takes a couple of minutes.

| Step | | |
|---|---|---|
| 1 | `put_events` | events into an event bank |
| 2 | `put_utdq_stations` | station locations |
| 3 | `put_utdq_picks` | picks + quality control |
| 4-5 | `build_manifests` | tables and metadata |
| 6 | `publish_network` | pull request on Hugging Face |

In [ ]:
!pip install -q utdquake cartopy   # cartopy: for the maps

In [ ]:
import os
os.environ["UTDQUAKE_ROOT"] = os.path.abspath("my_utdquake")   # where everything is built
UTDQ_NET = "DEMO"   # your network code: the agency code, or the next free R code (R14, ...)

## Your catalog
Any ObsPy `Catalog` with picks works. Here: 100 manually reviewed events of January 2021.

> ⚠️ **Depths must be below sea level** (QuakeML convention). Correct them first if your catalog uses the
> surface or a well head as reference.

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

cat = Client("http://coseismiq.ethz.ch:8080").get_events(
    contributor="ISOR_manual", starttime=UTCDateTime("2021-01-01"), endtime=UTCDateTime("2021-02-01"),
    limit=100, includearrivals=True)
print(len(cat), "events,", sum(len(e.picks) for e in cat), "picks")
print(cat[0])

## 1. Event bank

In [ ]:
from utdquake.bank.bank import UTDQBank
from utdquake.core.config import get_utdq_paths

ebank = UTDQBank(base_path=str(get_utdq_paths(UTDQ_NET)["banks"]),
                 path_structure="{year}/{month}/{day}", name_structure="{event_id_end}", format="quakeml")
ebank.put_events(cat)

## 2. Stations
One row per station: `network`, `station`, `latitude`, `longitude`, `elevation`.

In [ ]:
import obsplus

networks = sorted({p.waveform_id.network_code for e in cat for p in e.picks})
inv = Client("ETH").get_stations(network=",".join(networks), level="channel",
                                 starttime=UTCDateTime("2021-01-01"), endtime=UTCDateTime("2021-02-01"))
stations = obsplus.stations_to_df(inv).drop_duplicates(["network", "station"])[
    ["network", "station", "latitude", "longitude", "elevation"]]
print(networks, len(stations), "stations")

ebank.put_utdq_stations(stations, calculate_d_az=True)

## 3. Picks and quality control

In [ ]:
ebank.put_utdq_picks(apply_utdq_qc=True)

## 4-5. Tables and metadata
`catalog` and `catalog_type` (`agency`, `research` or `synthetic`) are **required**. `dataset_version` is set
for you: a new network goes into the next release (after `v2.1`, `v2.2`).

In [ ]:
import pandas as pd
from utdquake.writers.parquet import build_manifests, publish_flat_manifests

info = pd.DataFrame({
    "utdq_net": [UTDQ_NET],
    "catalog": ["COSEISMIQ"], "catalog_type": ["research"],
    "provider": ["COSEISMIQ / ETH Zurich"], "provider_url": ["http://coseismiq.ethz.ch"],
    "country": ["Iceland"], "continent": ["Europe"],
    "uploaded_by_github": ["https://github.com/<your-username>"],
})
build_manifests(utdq_nets=[UTDQ_NET], include_events=True, include_stations=True, include_picks=True,
                include_networks=True, overwrite=True, include_manual_network_info=info)
publish_flat_manifests()

## Your network
The same tables every UTDQuake network has:

In [ ]:
root = os.environ["UTDQUAKE_ROOT"]
events = pd.read_parquet(f"{root}/events/utdq_net={UTDQ_NET}.parquet")
stations = pd.read_parquet(f"{root}/stations/utdq_net={UTDQ_NET}.parquet")
picks = pd.read_parquet(f"{root}/picks/utdq_net={UTDQ_NET}.parquet")
print(f"{len(events)} events, {len(stations)} stations, {len(picks)} picks")
pd.read_parquet(f"{root}/network/network.parquet").T.head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sc = ax1.scatter(events.longitude, events.latitude, c=events.depth / 1e3, s=25, cmap="viridis_r")
ax1.scatter(stations.confirmed_longitude, stations.confirmed_latitude, marker="^", s=60,
            facecolor="none", edgecolor="k", label="stations")
fig.colorbar(sc, ax=ax1, label="depth (km)")
ax1.set(xlabel="longitude", ylabel="latitude", title="Hengill: earthquakes and stations")
ax1.legend()
for phase, color in (("P", "#1f77b4"), ("S", "#E87500")):
    p = picks[picks.phase.str.startswith(phase)]
    ax2.scatter(p.linear_hyp_distance, p.travel_time, s=8, alpha=0.5, color=color, label=phase)
ax2.set(xlabel="hypocentral distance (km)", ylabel="travel time (s)", title="Travel times")
ax2.legend()
plt.tight_layout()

## Check your network
Before publishing, make sure it looks as expected: the standard UTDQuake figures, and a travel-time model
fitted to your picks. Look for events in the right place and depth, stations where they should be, and
travel times on a clean trend (picks far from it have a high z-score).

In [ ]:
import utdquake as utdq
from utdquake.qc.travel_time import build_travel_time_model

build_travel_time_model(UTDQ_NET)   # also adds travel_time_zscore to the picks
net = utdq.Network(UTDQ_NET)
net.plot_overview()

In [ ]:
for phase in ("P", "S"):
    net.plot_travel_time_vs_distance_zscore(phase=phase, x_lim=(0, 30), y_lim=(0, 8), add_inset=False)

In [ ]:
net.plot_stats()

## 6. Publish
Opens a **pull request** on the [UTDQuake dataset](https://huggingface.co/datasets/ecastillot/UTDQuake);
the maintainer reviews and merges it. You need a
[Hugging Face token](https://huggingface.co/settings/tokens) and a real network code (not `DEMO`).

In [ ]:
# os.environ["HF_TOKEN"] = "hf_..."
# utdq.publish_network(UTDQ_NET, create_pr=True)

Publish the travel-time model too with `utdq.publish_network(UTDQ_NET, include_travel_time=True, create_pr=True)`.
**Optional:** `generate_network_figures` / `publish_network_figures` add your network to the
[overview](https://utdquake.readthedocs.io/en/latest/overview.html). See the
[upload guide](https://utdquake.readthedocs.io/en/latest/upload_dataset.html).

---
**Author:** Emmanuel Castillo, University of Texas at Dallas ·
[LinkedIn](https://www.linkedin.com/in/ecastillot/) · [castillo.280997@gmail.com](mailto:castillo.280997@gmail.com)